In [1]:
import os
os.environ['XLA_FLAGS'] = '--xla_force_host_platform_device_count=8'
import jax
import jax.numpy as jnp


In [4]:
from functools import partial

In [8]:
mesh = jax.sharding.Mesh(jax.devices(), ('d',))

In [ ]:
@jax.jit
def f(old_state, new_state, index):
    """
    old_state: [Total_B, 8] (sharded across 'd')
    new_state: [1, 8] (replicated)
    index: scalar (replicated)
    """
    @partial(
        jax.shard_map,
        mesh=mesh,
        in_specs=(jax.P('d'), jax.P(), jax.P()), 
        out_specs=jax.P('d')
    )
    def _f(old_state_shard, new_state_rep, index_rep):
        B = old_state_shard.shape[0] 
        device_id = jax.lax.axis_index('d')
        
        start_index = B * device_id
        end_index = B * (device_id + 1)

        local_idx = index_rep - start_index

        is_on_this_device = (index_rep >= start_index) & (index_rep < end_index)

        state = jax.lax.cond(
            is_on_this_device,
            lambda: old_state_shard.at[local_idx].set(new_state_rep.squeeze()),
            lambda: old_state_shard
        )
        return state

    return _f(old_state, new_state, index)

In [47]:
x = jnp.arange(16 * 8).reshape(16, 8)
y = jnp.arange(8).reshape(1, 8)
index = jnp.asarray(8, dtype='int32')

In [48]:
index

Array(8, dtype=int32)

In [49]:
out = f(x, y, index)

In [45]:
out

Array([[  0,   1,   2,   3,   4,   5,   6,   7],
       [  8,   9,  10,  11,  12,  13,  14,  15],
       [ 16,  17,  18,  19,  20,  21,  22,  23],
       [ 24,  25,  26,  27,  28,  29,  30,  31],
       [ 32,  33,  34,  35,  36,  37,  38,  39],
       [ 40,  41,  42,  43,  44,  45,  46,  47],
       [ 48,  49,  50,  51,  52,  53,  54,  55],
       [ 56,  57,  58,  59,  60,  61,  62,  63],
       [  0,   1,   2,   3,   4,   5,   6,   7],
       [ 72,  73,  74,  75,  76,  77,  78,  79],
       [ 80,  81,  82,  83,  84,  85,  86,  87],
       [ 88,  89,  90,  91,  92,  93,  94,  95],
       [ 96,  97,  98,  99, 100, 101, 102, 103],
       [104, 105, 106, 107, 108, 109, 110, 111],
       [112, 113, 114, 115, 116, 117, 118, 119],
       [120, 121, 122, 123, 124, 125, 126, 127]], dtype=int32)

In [30]:
from attr.validators import max_len


test_cache = KVCache(
    k=jnp.array([[0,0,0,1,2,3,4,5], [0,0,0,0,0,1,2,3]]),
    length=2, 
    max_len=7
)



In [ ]:

@jax.jit
def roll_cache(cache):

    max_seq_len = cache.max_len
    length = cache.length
    new_k = jnp.roll(cache.k, shift=-(cache.length - cache.max_len), axis=-1)

    return KVCache(
        k=new_k,
        length=cache.max_len,
        max_len=cache.max_len
    )

@jax.jit 
def roll_length(cache):
    



In [31]:
shift_pad(test_cache)

KVCache(k=Array([[1, 2, 3, 4, 5, 0, 0, 0],
       [0, 0, 1, 2, 3, 0, 0, 0]], dtype=int32), length=Array(7, dtype=int32, weak_type=True), max_len=Array(7, dtype=int32, weak_type=True))